# 10 – Pipeline per Konfigurationsdatei (Etappe 9, Stufe 2)

**Was:** Die ganze Kette mit `astropipe.run` aus einer TOML-Datei ausführen – für NGC 3132 und NGC 6720 (NIRCam) – und mit den Ergebnissen der Notebooks vergleichen.

**Warum:** Ein Skript ist nur dann ein Ersatz für die Notebooks, wenn es mit denselben Parametern dasselbe Bild liefert (Regressionstest, wie in Notebook 07). Erst danach bauen wir darauf den Entdecker-Modus auf.

**Im Terminal geht dasselbe mit:**
```bash
python -m astropipe.run configs/ngc3132.toml
```
Zwischenergebnisse landen in `data/<name>_run/`, Bilder in `output/runs/` (Web-JPEG + Bildunterschrift) und `output/full_res/`, alle verwendeten Werte in `params/<name>_run.json`.

**Ergebnis:** _(nach dem Ausführen eintragen)_

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import numpy as np
from astropy.io import fits
from astropipe import run

ROOT = Path("..")

def report(name, a, b, tol=1e-5):
    d = np.nanmax(np.abs(np.asarray(a, np.float64) - np.asarray(b, np.float64)))
    print(f"{name:<30} max |diff| = {d:.2e}  {'OK' if d < tol else 'CHECK'}")

## 1. NGC 3132

Erwartung: Streckung und Farbbild C identisch zu den Notebooks 04/05 (Farbe dort mit exakten Farbwinkeln berechnet, hier ebenfalls).

In [2]:
info = run.run(ROOT / "configs" / "ngc3132.toml", root=ROOT)

[12:15:57] MAST: 6 i2d products, 6 selected (2.49 GB)
6 of 6 files already present, 0 to download (nan GB)
[12:15:58] target grid (3127, 3121)
F090W   exists -> ngc3132_f090w_reproj.fits
F187N   exists -> ngc3132_f187n_reproj.fits
F212N   exists -> ngc3132_f212n_reproj.fits
F356W   exists -> ngc3132_f356w_reproj.fits
F405N   exists -> ngc3132_f405n_reproj.fits
F470N   exists -> ngc3132_f470n_reproj.fits


/Users/inu/Projects/astro-pipeline/src/astropipe/mast.py:28: UserWarning: Warning: converting a masked element to nan.
  return float(products["size"].sum()) / 1e9


[12:15:58] star offsets vs F470N: {'F090W': 0.36, 'F187N': 0.19, 'F212N': 0.32, 'F356W': 0.24, 'F405N': 0.42, 'F470N': 0.0}
[12:16:00] 6 sky boxes
[12:16:04] crop 1578 x 1578 px = 1.66' x 1.66'
[12:16:04] exported C_physical
[12:16:04] done in 8.4 s -> params/ngc3132_run.json


In [3]:
for b in ["F090W", "F187N", "F212N", "F356W", "F405N", "F470N"]:
    new = fits.getdata(ROOT / "data" / "ngc3132_run" / f"ngc3132_{b.lower()}_stretched.fits")
    old = fits.getdata(ROOT / "data" / "ngc3132_nircam_stretched" / f"ngc3132_{b.lower()}_stretched.fits")
    report(f"stretched {b}", new, old)
report("colour C_physical",
       np.load(ROOT / "data" / "ngc3132_run" / "ngc3132_C_physical.npy"),
       np.load(ROOT / "data" / "ngc3132_nircam_color" / "ngc3132_C_physical.npy"))
print("crop:", info["export"]["center_yx"], "half (h, w):", info["export"]["half_hw"], "(notebook 06: centre (1570, 1575), half 789)")

stretched F090W                max |diff| = 0.00e+00  OK
stretched F187N                max |diff| = 0.00e+00  OK
stretched F212N                max |diff| = 0.00e+00  OK
stretched F356W                max |diff| = 0.00e+00  OK
stretched F405N                max |diff| = 0.00e+00  OK
stretched F470N                max |diff| = 0.00e+00  OK
colour C_physical              max |diff| = 0.00e+00  OK
crop: [1570, 1575] half (h, w): [789, 789] (notebook 06: centre (1570, 1575), half 789)


## 2. NGC 6720

Mit lokaler Streifenkorrektur (`clean.method = "local"`). Erwartung: Streckung identisch zu Notebook 08 (Version `lw_dsl`), Ausschnitt wie dort: Mitte (1671, 1608), 1562 × 1172 px. Die Streifenkorrektur rechnet beim ersten Lauf einige Minuten.

In [4]:
info2 = run.run(ROOT / "configs" / "ngc6720.toml", root=ROOT)

[12:16:23] MAST: 4 i2d products, 4 selected (1.79 GB)
4 of 4 files already present, 0 to download (nan GB)


/Users/inu/Projects/astro-pipeline/src/astropipe/mast.py:28: UserWarning: Warning: converting a masked element to nan.
  return float(products["size"].sum()) / 1e9


[12:16:27] destriped F162M (local)
[12:16:31] destriped F212N (local)
[12:16:32] destriped F300M (local)
[12:16:33] destriped F335M (local)
[12:16:33] target grid (3311, 3312)
F162M     23.6 s -> ngc6720_f162m_reproj.fits
F212N     23.7 s -> ngc6720_f212n_reproj.fits
F300M      9.2 s -> ngc6720_f300m_reproj.fits
F335M      9.2 s -> ngc6720_f335m_reproj.fits
[12:17:39] star offsets vs F335M: {'F162M': 0.16, 'F212N': 0.05, 'F300M': 0.02, 'F335M': 0.0}
[12:17:41] 4 sky boxes
[12:17:44] crop 1562 x 1172 px = 1.64' x 1.23'
[12:17:44] exported C_physical
[12:17:44] done in 89.1 s -> params/ngc6720_run.json


In [5]:
for b in ["F162M", "F212N", "F300M", "F335M"]:
    new = fits.getdata(ROOT / "data" / "ngc6720_run" / f"ngc6720_{b.lower()}_stretched.fits")
    old = fits.getdata(ROOT / "data" / "ngc6720_nircam_stretched" / f"ngc6720_{b.lower()}_stretched.fits")
    report(f"stretched {b}", new, old)
e = info2["export"]
print("crop centre:", e["center_yx"], "size px:", 2 * e["half_hw"][1], "x", 2 * e["half_hw"][0])
print("sky boxes used:", info2["stretch"]["sky_boxes"]["yx"])

stretched F162M                max |diff| = 0.00e+00  OK
stretched F212N                max |diff| = 0.00e+00  OK
stretched F300M                max |diff| = 0.00e+00  OK
stretched F335M                max |diff| = 0.00e+00  OK
crop centre: [1671, 1608] size px: 1562 x 1172
sky boxes used: [[1125, 2325], [2100, 2175], [450, 1575], [2400, 1125]]


## Fazit

- `python -m astropipe.run configs/<name>.toml` führt die ganze Kette aus (Download, optionale Streifenkorrektur, Registrierung, Hintergrund, Streckung, Farbe, Ausschnitt, Export mit Bildunterschrift) und protokolliert alle Werte in `params/<name>_run.json`.
- **Regressionstest bestanden:** NGC 3132 und NGC 6720 liefern **bitgenau** dieselben gestreckten Bilder wie die Notebooks, NGC 3132 auch dasselbe Farbbild (Farbwinkel statt gerundeter RGB-Werte → exakt). Ausschnitte und Himmelsboxen identisch.
- **Laufzeit:** wenige Sekunden mit vorhandenen Zwischenergebnissen, ≈ 1,5 min für NGC 6720 von den Rohdaten an.
- Auch im Terminal getestet.
- **Noch nicht im Skript:** NIRCam + MIRI (Instrumentenversatz, LRGB, Schärfe angleichen) → optionale Stufe 2b.